## Middleware

- These are basically hooks which can act as a trigger for a tool call or a function.
- Can be used to control what happens inside an agent.
- The hooks can be retries, fall backs, rate limits, tranforming prompts, tool selection and output formatting etc.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

There are few in-built hooks too like 
- Summarization
- Human In Feedback
... more in Langchain

### Summarization
automatically summarize when approaching token limits

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",  # We can use flash or mini models for summarization to save tokens
            trigger=("messages", 10),  # At every 10 messages summarization will work.
            keep=("messages", 4),  # To have enough context keep last 4 chats
        )
    ],
)

### What is a thread?

A thread is one conversation, identified by a `thread_id`. It is the key the `checkpointer` stores state under.

- Same `thread_id`: loads the saved messages, appends the new turn, saves again. This is what makes the agent remember.
- New `thread_id`: fresh conversation, empty state.
- No `config`: error, since the checkpointer has no key to save under.

The agent itself is stateless. Memory lives in the checkpointer, so one agent can serve many conversations (one `thread_id` per user).

`InMemorySaver` keeps state in RAM, so it is lost when the kernel restarts. Use `SqliteSaver` or `PostgresSaver` to persist.

In [3]:
config = {"configurable": {"thread_id": "testing-01"}}

In [4]:
import re
from IPython.display import Markdown, display


def clean(text):
    """qwen wraps its reasoning in <think> tags, which Markdown hides as HTML."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


def get_summary(messages):
    """The middleware injects the summary as a HumanMessage tagged lc_source."""
    return next(
        (
            m.content
            for m in messages
            if m.additional_kwargs.get("lc_source") == "summarization"
        ),
        None,
    )


def ask(agent, questions, config):
    last_summary = None
    for q in questions:
        messages = agent.invoke({"messages": [HumanMessage(content=q)]}, config)[
            "messages"
        ]

        display(Markdown(f"**Q:** {q}"))
        display(Markdown(clean(messages[-1].content)))

        summary = get_summary(messages)
        if summary and summary != last_summary:
            display(Markdown(f"**Summary injected:**\n\n{clean(summary)}"))
            last_summary = summary

        print(f"{len(messages)} messages in state\n")


questions_list = [
    "In one sentence, what is a vector database?",
    "Name two popular ones.",
    "Which one is easiest to run locally?",
    "How is it different from a normal SQL database?",
    "What is an embedding?",
    "Which model would you use to create embeddings?",
    "What does cosine similarity measure?",
    "Now, what was the very first thing I asked you about?",
]

ask(agent, questions_list, config)

**Q:** In one sentence, what is a vector database?

A vector database is a specialized data storage system optimized for efficiently indexing, storing, and searching high-dimensional numerical representations (vectors) to enable fast similarity matching for AI and machine learning applications.

2 messages in state



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** (fully managed) and **Milvus** (open-source).

...

Cosine similarity measures the **directional alignment** between two vectors in multi-dimensional space, completely ignoring their magnitude (length). It calculates the cosine of the angle between them:

- **1.0** → Vectors point in the exact same direction (maximally similar)
- **0.0** → Vectors are orthogonal (no shared direction / unrelated)
- **-1.0** → Vectors point in exactly opposite directions (maximally dissimilar)

### 🔑 Why it matters for embeddings
Embedding models place semantically similar concepts in similar directions in vector space. Cosine similarity captures that **semantic alignment** regardless of how "long" or "short" the vectors are. For example, a 10-word query and a 500-word document can score `0.92` cosine similarity if they share the same underlying meaning, even though their vector lengths differ drastically.

### 📐 The math (simplified)
```
cos(θ) = (A · B) / (||A|| × ||B||)
```
Where `A · B` is the dot product and `||A||`, `||B||` are the magnitudes.

### 💡 Practical note
Most modern embedding models **normalize vectors to unit length** (`||v|| = 1`). When normalized, cosine similarity simplifies to just the dot product, and the range effectively becomes **0 to 1**. This is why vector databases and retrieval libraries often report similarity scores between 0 and 1.

### 🆚 Cosine vs. Euclidean distance
- **Cosine**: Cares only about direction → best for semantic/text embeddings
- **Euclidean**: Cares about direction + magnitude → can unfairly penalize longer documents or higher-magnitude vectors

*(Other metrics like dot product, Manhattan, or Jaccard exist, but cosine remains the industry standard for text/image embeddings.)*

Want a quick Python snippet showing how to compute it, or how vector databases use it under the hood?

8 messages in state



**Q:** Now, what was the very first thing I asked you about?

The very first thing you asked about was **how a vector database differs from a normal SQL database**.

10 messages in state



In [5]:
# What is stored in this thread right now. No API call.
for m in agent.get_state(config).values["messages"]:
    tag = m.additional_kwargs.get("lc_source", "")
    print(f"{type(m).__name__:12} {tag:14} {clean(m.content)[:70]!r}")

HumanMessage summarization  'Here is a summary of the conversation to date:\n\n## SESSION INTENT\nExpl'
AIMessage                   'A vector database differs from a traditional SQL database in four fund'
HumanMessage                'What is an embedding?'
AIMessage                   'An embedding is a fixed-length numerical vector generated by a machine'
HumanMessage                'Which model would you use to create embeddings?'
AIMessage                   'The best model depends on your data type, deployment constraints, and '
HumanMessage                'What does cosine similarity measure?'
AIMessage                   'Cosine similarity measures the **directional alignment** between two v'
HumanMessage                'Now, what was the very first thing I asked you about?'
AIMessage                   'The very first thing you asked about was **how a vector database diffe'


In [6]:
# We can also trigger on tokens instead of message count.

token_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",
            trigger=("tokens", 500),  # Summarize once history passes 500 tokens.
            keep=("tokens", 200),  # Keep roughly the last 200 tokens.
        )
    ],
)

token_config = {"configurable": {"thread_id": "tokens-01"}}

ask(token_agent, questions_list[:5], token_config)

**Q:** In one sentence, what is a vector database?

A vector database is a specialized database that stores data as high-dimensional numerical vectors and enables efficient similarity search to retrieve items based on their semantic or mathematical proximity to a query.

2 messages in state



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Qdrant**.

...

3 messages in state



**Q:** What is an embedding?

In machine learning, an **embedding** is a way of converting discrete, non-numeric data (like words, sentences, images, users, or products) into a **dense list of numbers** (a vector) that captures its meaning and relationships.

### How it works (conceptually)
Instead of treating each item as an isolated symbol, an embedding places it in a multi-dimensional mathematical space. Items that are semantically or functionally similar end up **closer together** in that space, while unrelated items are farther apart.

**Example (text):**  
The embeddings for `"king"` and `"queen"` will be near each other, just as `"cat"` and `"dog"` will cluster together. The model learns these positions by analyzing patterns in large datasets, often capturing intuitive relationships like:  
`embedding("king") - embedding("man") + embedding("woman") ≈ embedding("queen")`

### Why embeddings are useful
- **Machines need numbers:** Neural networks can't directly process raw text, images, or IDs. Embeddings turn them into numerical form.
- **Capture context & similarity:** They preserve semantic meaning, enabling tasks like finding related documents, recommending similar products, or clustering data.
- **Efficient computation:** Comparing embeddings (e.g., using cosine similarity) is fast and scales well for search, retrieval, and matching.

### Common use cases
- 📖 **NLP:** Word, sentence, or document embeddings (e.g., Word2Vec, BERT, OpenAI's text-embedding models)
- 🔍 **Semantic search & RAG:** Finding relevant context for LLMs
- 🛒 **Recommendations:** Matching users to products based on behavior embeddings
- 🖼️ **Multimodal AI:** Image, audio, or video embeddings for cross-modal retrieval

### Technical note
An embedding is typically a vector of real numbers with fixed dimensions (e.g., 384, 768, or 1536). Each individual dimension doesn't usually have a human-readable meaning, but together they encode rich, learnable patterns optimized for downstream tasks.

Let me know if you'd like a code example or a deeper dive into how they're trained or used in practice!

**Summary injected:**

Here is a summary of the conversation to date:

Error generating summary: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kxn03mk2frjb59vv2wgvmjfc` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 10377, Requested 2628. Please try again in 5.025s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'compound', 'code': 'rate_limit_exceeded'}}

3 messages in state



In [7]:
# Or trigger on a fraction of the model's context window, which adapts to the model.

fraction_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",
            trigger=("fraction", 0.005),  # Summarize at 0.5% of the context window.
            keep=("fraction", 0.002),  # Keep the last 0.2%.
        )
    ],
)

fraction_config = {"configurable": {"thread_id": "fraction-01"}}

ask(fraction_agent, questions_list[:4], fraction_config)

**Q:** In one sentence, what is a vector database?

A vector database is a specialized data store optimized to index and rapidly retrieve high-dimensional numerical embeddings using distance metrics for fast similarity search and AI-driven applications.

2 messages in state



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Weaviate**.

...

5 messages in state



**Q:** How is it different from a normal SQL database?

The primary difference lies in **how they store data** and **how they find it**.

Here is the breakdown:

### 1. Data Structure
*   **SQL Database:** Stores **structured data** in tables (rows and columns). It is rigid; you must define the schema (e.g., `User_ID`, `Name`, `Email`) before inserting data.
*   **Vector Database:** Stores **high-dimensional vectors** (long lists of numbers/floats). It is schema-flexible and designed to hold "embeddings"—mathematical representations of unstructured data like text, images, or audio.

### 2. Search Logic (The Big Difference)
*   **SQL (Exact Match):** Optimized for finding **exact** matches or strict ranges.
    *   *Query:* "Find the user whose name is **exactly** 'Alice'."
    *   *Math:* Equality checks (`=`), greater than/less than (`>`, `<`).
*   **Vector DB (Similarity):** Optimized for finding **similar** concepts.
    *   *Query:* "Find the user whose profile is **most similar** to 'Alice'."
    *   *Math:* Distance metrics (Cosine similarity, Euclidean distance) to calculate how close two vectors are in multi-dimensional space.

### 3. Indexing
*   **SQL:** Uses **B-Trees**. This allows it to jump instantly to a specific row based on a unique key (like a phonebook index).
*   **Vector DB:** Uses **Approximate Nearest Neighbor (ANN)** algorithms (like HNSW or IVF). Since comparing billions of vectors is too slow for exact math, these algorithms approximate the "closest" neighbors to return results in milliseconds.

### 4. Primary Use Case
*   **SQL:** Accounting, inventory, user management, transaction logs (ACID compliance is key).
*   **Vector DB:** AI applications, semantic search (Google/Bing), recommendation engines (Netflix/Spotify), and image recognition.

### Summary Analogy
*   **SQL** is like a **library card catalog**. You find a book because you know the exact title or author.
*   **Vector DB** is like a **knowledgeable librarian**. You say, "I want a book that feels like *The Matrix* but takes place in space," and they find the closest thematic match, even if the titles don't match exactly.

**Note:** They are often used together. A modern AI app might use **SQL** to store the user's login info and **Vector DB** to store the semantic understanding of the documents the user is searching.

**Summary injected:**

Here is a summary of the conversation to date:

## SESSION INTENT
Provide a concise definition of a vector database, name two popular vector databases, and identify which of those is easiest to run locally.

## SUMMARY
- Defined a vector database in one sentence: “A vector database is a specialized data store optimized to index and rapidly retrieve high-dimensional numerical embeddings using distance metrics for fast similarity search and AI‑driven applications.”
- Named two popular vector databases: Pinecone and Weaviate.
- Determined that Weaviate is the easiest to run locally (open‑source, Docker/pip/standalone options), while Pinecone is a fully managed cloud service with no official self‑hosted version. Included a quick Docker command example for Weaviate.

## ARTIFACTS
None.

## NEXT STEPS
Await further user queries or instructions.

3 messages in state



## Human In Loop Middleware

Pause agent execution for human approval, editing or rejection.
Usecase:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [8]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email. Mock implementation."""
    return f"Email sent to {recipient} successfully."


def read_email_tool(email_id: str) -> str:
    """Read an email by id. Mock implementation."""
    return "Email content: [content-here]"

In [9]:
hitl_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[send_email_tool, read_email_tool],  # Tools must be registered to be called.
    checkpointer=InMemorySaver(),  # Required, since an interrupt has to be resumed later.
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
                "read_email_tool": False,  # False means run it without asking.
            }
        )
    ],
)

In [10]:
# Step 1: the agent pauses instead of sending. Note the __interrupt__ key.

approve_config = {"configurable": {"thread_id": "test-approve"}}

result = hitl_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    approve_config,
)

request = result["__interrupt__"][0].value
for action in request["action_requests"]:
    print(action["name"], action["args"])

send_email_tool {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}


In [11]:
# Step 2: approve. The tool runs and the agent continues from where it paused.

result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    approve_config,
)
print(clean(result["messages"][-1].content))

The email has been sent to john@test.com successfully.


In [12]:
# Rejecting instead. The message tells the model why, so it can adapt.

reject_config = {"configurable": {"thread_id": "test-reject"}}

hitl_agent.invoke(
    {"messages": [HumanMessage(content="Send an email to spam@test.com saying 'buy now'")]},
    reject_config,
)
result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [{"type": "reject", "message": "Do not send marketing email."}]
        }
    ),
    reject_config,
)
print(clean(result["messages"][-1].content))

I cannot send this email. The system returned an error message: "Do not send marketing email."


In [13]:
# read_email_tool was set to False, so it runs with no interrupt at all.

read_config = {"configurable": {"thread_id": "test-read"}}

result = hitl_agent.invoke(
    {"messages": [HumanMessage(content="Read email with id abc-123")]},
    read_config,
)
print("interrupted:", "__interrupt__" in result)
print(clean(result["messages"][-1].content))

interrupted: False
Here is the content of the email with id abc-123:

Email content: [content-here]
